# HDB Resale Flat Prices ETL — Part 1

Part 1 of the HDB Senior Data Engineer technical test. ETL pipeline producing a cleaned master dataset of HDB resale flat transactions covering **January 2012 – December 2016**, sourced from [data.gov.sg collection 189](https://data.gov.sg/collections/189/view).

## How to use this notebook

Run the cells top to bottom. On a fresh clone the raw CSVs are already present in `data/raw/`, so the pipeline runs end-to-end without network access. The download path is still exercised when run, and is idempotent: files whose size matches the API's `datasetSize` are skipped.

All pipeline logic lives in `src/`. The notebook imports it and orchestrates each stage; heavy logic is intentionally kept out of cells.

## Stages mapped to the brief

| Brief requirement | Notebook section | Status |
|---|---|---|
| Dataset extraction (Jan 2012 – Dec 2016) | §1 Ingest | implemented |
| Combine into a single master dataset (DQ §1) | §2 Combine | implemented |
| Data profiling (DQ §2) | §3 Profile | implemented |
| Validation rules on Date/Town/Flat Type/Flat Model/storey_range (DQ §3) | §4 Validate | pending |
| Recompute `remaining_lease` as of today (DQ §4) | §5 Clean | pending |
| Dedupe on composite key, keep highest price (DQ §5) | §5 Clean | pending |
| Anomaly heuristic on resale price (DQ §6) | §5 Clean | pending |
| Resale Identifier + hashing (Transformation §1–3) | — | **out of scope this pass** |

## 0. Setup

Configure logging and import the pipeline modules. Editable install (`pip install -e ".[dev]"`) makes `from src...` resolvable from the notebook working directory.

In [1]:
import logging
import sys
from pathlib import Path

# Make `from src import ...` resolvable regardless of how Jupyter was
# launched. The editable install (`pip install -e ".[dev]"`) handles this
# when the kernel runs inside the project venv, but Jupyter sometimes
# picks up a different `python` kernel (e.g. a global ipykernel) where
# `src` is not on the path. Inserting the project root makes the notebook
# robust to that — reviewers don't need to fight kernel.json.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import pyarrow
import requests

# Environment sanity check — fails loudly here on a misconfigured venv
# rather than mid-pipeline. If any of these imports raise, the reviewer
# almost certainly skipped `pip install -e ".[dev]"` from the project root.
print(f"python:   {sys.version.split()[0]}")
print(f"pandas:   {pd.__version__}")
print(f"requests: {requests.__version__}")
print(f"pyarrow:  {pyarrow.__version__}")

# INFO-level surfaces the per-stage progress lines emitted by src/ingest.py
# and src/combine.py directly into the notebook output.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

from src import config
from src.ingest import ingest_all, verify_raw
from src.combine import combine_raw_files

print()
print("project root:", config.PROJECT_ROOT)
print("scope window:", config.SCOPE_START, "..", config.SCOPE_END)
print("as_of_date:  ", config.AS_OF_DATE)

python:   3.12.2
pandas:   3.0.2
requests: 2.33.1
pyarrow:  23.0.1

project root: /Users/ivanong/Documents/GitHub/hdb_resale_prices
scope window: 2012-01 .. 2016-12
as_of_date:   2026-04-09


## 1. Ingest

**Brief reference:** *Dataset Extraction* — *"Your ETL pipeline should process the data file as it is, without any manual modifications."*

**Goal:** Download the in-scope HDB resale CSVs from data.gov.sg into `data/raw/`.

**Approach:** Discover dataset IDs at runtime by walking the collection metadata endpoint and filtering on coverage window — no dataset IDs or filenames are hardcoded, in line with the brief's *"avoid hardcoding where possible"* directive. Downloads stream from the pre-signed S3 URLs returned by the `poll-download` endpoint, with 429 retry/backoff and `datasetSize`-based idempotency.

**Robustness:** If the API is unreachable, the pipeline falls back to whatever CSVs are already present in `data/raw/`. Because raw data is committed to the repo, the notebook is runnable on a fresh clone with no network access.

In [2]:
downloaded_paths = ingest_all()

print()
print(f"{len(downloaded_paths)} files in data/raw/:")
for p in downloaded_paths:
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")

2026-04-09 10:21:47,828 INFO src.ingest: Collection 189 has 5 child datasets


2026-04-09 10:21:49,307 INFO src.ingest: Filtered to 3 in-scope datasets for 2012-01..2016-12


2026-04-09 10:21:49,505 INFO src.ingest: poll-download rate-limited for d_43f493c6c50d54243cc1eab0df142d6a; sleeping 3.0s


2026-04-09 10:21:52,698 INFO src.ingest: poll-download rate-limited for d_43f493c6c50d54243cc1eab0df142d6a; sleeping 3.0s


2026-04-09 10:21:55,983 INFO src.ingest: poll-download rate-limited for d_43f493c6c50d54243cc1eab0df142d6a; sleeping 3.0s


...

2026-04-09 10:22:10,421 INFO src.ingest: poll-download rate-limited for d_ea9ed51da2787afaf8e51f827c304208; sleeping 3.0s


2026-04-09 10:22:13,630 INFO src.ingest: poll-download rate-limited for d_ea9ed51da2787afaf8e51f827c304208; sleeping 3.0s


2026-04-09 10:22:16,829 INFO src.ingest: poll-download rate-limited for d_ea9ed51da2787afaf8e51f827c304208; sleeping 3.0s


2026-04-09 10:22:20,293 INFO src.ingest: Skip ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv (already present, 3070924 bytes)



3 files in data/raw/:
  ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv  (29,369,945 bytes)
  ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv  (4,160,771 bytes)
  ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv  (3,070,924 bytes)


### 1.1 Verify

For each CSV in `data/raw/`, parse it with pandas and check three things against the API's authoritative metadata:

| Check | What it confirms |
|---|---|
| **size_ok** | File size equals the API's `datasetSize` exactly |
| **cols_ok** | Column names equal `columnMetadata` (in order) |
| **dates_ok** | Every row's `month` falls inside the dataset's coverage window |

Together these form the strongest integrity envelope available without a server-published checksum. `verify_raw()` also reports any in-scope dataset that has no local file, so the inverse failure mode is caught too.

In [3]:
verification = verify_raw()
verification_df = pd.DataFrame(verification)

display(verification_df[[
    "file", "matched_dataset", "rows", "cols",
    "min_month", "max_month",
    "actual_size", "expected_size",
    "size_ok", "cols_ok", "dates_ok", "ok",
]])

assert all(r["ok"] for r in verification), "Stage 1 verification failed — see logs above"

2026-04-09 10:22:20,536 INFO src.ingest: Collection 189 has 5 child datasets


2026-04-09 10:22:22,028 INFO src.ingest: Filtered to 3 in-scope datasets for 2012-01..2016-12


2026-04-09 10:22:22,317 INFO src.ingest: [OK] ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv  rows=369651 cols=10  2000-01..2012-02  size_ok=True cols_ok=True dates_ok=True


2026-04-09 10:22:22,350 INFO src.ingest: [OK] ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv  rows=37153 cols=11  2015-01..2016-12  size_ok=True cols_ok=True dates_ok=True


2026-04-09 10:22:22,392 INFO src.ingest: [OK] ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv  rows=52203 cols=10  2012-03..2014-12  size_ok=True cols_ok=True dates_ok=True


2026-04-09 10:22:22,392 INFO src.ingest: verify_raw: 3 files checked, overall OK


,file,matched_dataset,rows,cols,min_month,max_month,actual_size,expected_size,size_ok,cols_ok,dates_ok,ok
0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,d_43f493c6c50d54243cc1eab0df142d6a,369651,10,2000-01,2012-02,29369945,29369945,True,True,True,True
1,ResaleFlatPricesBasedonRegistrationDateFromJan...,d_ea9ed51da2787afaf8e51f827c304208,37153,11,2015-01,2016-12,3070924,3070924,True,True,True,True
2,ResaleFlatPricesBasedonRegistrationDateFromMar...,d_2d5ff9ea31397b66239f245f57751537,52203,10,2012-03,2014-12,4160771,4160771,True,True,True,True


## 2. Combine

**Brief reference:** *Data Quality Requirements §1* — *"Combine the datasets into a single master dataset. The combined dataset should contain all the attributes found in all files."*

**Goal:** Union the verified raw CSVs into a single master DataFrame ready for profiling, validation, and cleaning.

**Per-file pipeline:**

1. **Normalize headers** — `strip().lower().replace(" ", "_")` on every column on read. Defends against case/whitespace drift across vintages so the schema union doesn't produce duplicates like `Month` vs `month`.
2. **Filter to scope** — drop rows whose `month` falls outside `SCOPE_START..SCOPE_END`. For the 2000–Feb 2012 file this trims to its Jan–Feb 2012 tail; for the other two it's a no-op. Applying it uniformly avoids special-casing.
3. **Normalize `remaining_lease`** — the Jan 2015–Dec 2016 file stores it as integer years already (verified by inspection: dtype `int64`, sample values like 70, 65, 64). We copy it into a canonical `remaining_lease_years_original` `Int64` column. Pre-2015 vintages don't have this column at all, so the master will hold `NaN` for those rows. The 2017+ "X years Y months" string format is out of scope and not parsed here.
4. **Tag with `source_file`** — lineage column for downstream debugging.

Frames are then concatenated with `sort=False`, taking the union of columns (pandas fills `NaN` where a column is absent in any particular frame).

In [4]:
master = combine_raw_files()

print()
print(f"Master shape: {master.shape[0]:,} rows x {master.shape[1]} columns")
print()
print("Rows per source file:")
print(master.groupby("source_file").size().to_string())
print()
print(f"Month range: {master['month'].min()} .. {master['month'].max()}")
print()
print("remaining_lease_years_original by source file:")
print(
    master.groupby("source_file")["remaining_lease_years_original"]
    .apply(lambda s: f"{s.notna().sum():>6,d} / {len(s):>6,d} populated")
    .to_string()
)

2026-04-09 10:22:22,665 INFO src.combine: ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv: 369651 rows raw, 3188 in scope (2012-01..2016-12)


2026-04-09 10:22:22,698 INFO src.combine: ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv: 37153 rows raw, 37153 in scope (2012-01..2016-12)


2026-04-09 10:22:22,740 INFO src.combine: ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv: 52203 rows raw, 52203 in scope (2012-01..2016-12)


2026-04-09 10:22:22,745 INFO src.combine: Combined master: 92544 rows, 13 columns from 3 files



Master shape: 92,544 rows x 13 columns

Rows per source file:
source_file
ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv                  3188
ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv    37153
ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv    52203

Month range: 2012-01 .. 2016-12

remaining_lease_years_original by source file:
source_file
ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv                      0 /  3,188 populated
ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv    37,153 / 37,153 populated
ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv         0 / 52,203 populated


### 2.1 Sample

First five rows of the combined master, with all 13 columns:

In [5]:
display(master.head())

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,source_file,remaining_lease,remaining_lease_years_original
0,2012-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,257800.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>
1,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,263000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>
2,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,275000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>
3,2012-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,260000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>
4,2012-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,226000.0,ResaleFlatPricesBasedonApprovalDate2000Feb2012...,NaN,<NA>


## 3. Profile

**Brief reference:** *Data Quality Requirements §2* — *"Perform data profiling on the master dataset."*

**Goal:** Surface the structural and statistical properties of the master so the next stages (validation rules, cleaning policies, anomaly thresholds) are informed by what's actually in the data, not assumed.

**Approach:** Hand-rolled profiling in `src/profile.py` (no `ydata-profiling`). The output is a `ProfileReport` dataclass holding pre-built DataFrames so each subsection below renders with a single `display(...)` call. Sections:

* **3.1 Overview** — per-column dtype/null/unique/sample
* **3.2 Numeric stats** — min/max/mean/median/std/quartiles
* **3.3 String columns** — length distribution and top-N value counts
* **3.4 Flagged columns** — high-null, constant, and whitespace contamination

The function is pure — it does not mutate the master.

In [6]:
from src.profile import profile_dataset

report = profile_dataset(master)

print(f"Rows x cols: {report.shape[0]:,} x {report.shape[1]}")
print(f"Month range: {report.month_range[0]} .. {report.month_range[1]}")
print(f"Numeric columns:    {len(report.numeric)}")
print(f"String columns:     {len(report.string_lengths)} (excluding source_file)")
print(f"Flagged columns:    {report.flags['column'].nunique() if not report.flags.empty else 0}")

Rows x cols: 92,544 x 13
Month range: 2012-01 .. 2016-12
Numeric columns:    5
String columns:     7 (excluding source_file)
Flagged columns:    2


### 3.1 Per-column overview

One row per column: dtype, null count and percentage, distinct value count, and a short sample of distinct non-null values. The `unique_count` here is what Stage 4 will use to size each rule's allowed set.

In [7]:
display(report.overview)

,column,dtype,null_count,null_pct,unique_count,sample_values
0,month,str,0,0.000000,60,"'2012-01', '2012-02', '2015-01'"
1,town,str,0,0.000000,26,"'ANG MO KIO', 'BEDOK', 'BISHAN'"
2,flat_type,str,0,0.000000,7,"'2 ROOM', '3 ROOM', '4 ROOM'"
3,block,str,0,0.000000,2139,"'406', '314', '170'"
4,street_name,str,0,0.000000,522,"'ANG MO KIO AVE 10', 'ANG MO KIO AVE 3', 'ANG ..."
5,storey_range,str,0,0.000000,25,"'01 TO 03', '07 TO 09', '10 TO 12'"
6,floor_area_sqm,float64,0,0.000000,168,"44.0, 45.0, 60.0"
7,flat_model,str,0,0.000000,20,"'Improved', 'New Generation', 'Model A'"
8,lease_commence_date,int64,0,0.000000,48,"1979, 1978, 1986"
9,resale_price,float64,0,0.000000,2615,"257800.0, 263000.0, 275000.0"


### 3.2 Numeric column statistics

`min`, `max`, `mean`, `median`, `std`, and quartiles for every numeric column. These feed Stage 5's anomaly heuristic (asymmetric IQR on price-per-sqm) and let us sanity-check that `floor_area_sqm`, `lease_commence_date`, and `resale_price` are within plausible HDB ranges.

In [8]:
display(report.numeric)

,min,max,mean,median,std,q1,q3
floor_area_sqm,31.0,280.0,96.569115,95.0,24.682292,74.0,111.0
lease_commence_date,1966.0,2013.0,1990.072701,1988.0,10.446719,1983.0,1999.0
resale_price,190000.0,1150000.0,450938.971730,428000.0,128181.301162,357000.0,515000.0
remaining_lease,48.0,97.0,73.913116,72.0,10.885456,66.0,83.0
remaining_lease_years_original,48.0,97.0,73.913116,72.0,10.885456,66.0,83.0


### 3.3 String columns — lengths and top values

Length stats catch unexpectedly long or zero-length values; top-N value counts give a feel for cardinality and the dominant categories that Stage 4's validation rules will derive their allowed sets from.

In [9]:
display(report.string_lengths)

# Top-N value counts per string column. `source_file` is excluded by design
# (lineage metadata, has exactly one value per source file).
for col, vc in report.string_top_values.items():
    print(f"\n--- top {len(vc)} values: {col} ---")
    display(vc)

,column,min_len,max_len,mean_len
0,month,7,7,7.000000
1,town,5,15,9.095468
2,flat_type,6,16,6.239400
3,block,1,4,2.995278
4,street_name,7,20,14.055746
5,storey_range,8,8,8.000000
6,flat_model,4,22,9.813300



--- top 10 values: month ---


,month,count
0,2012-03,2360
1,2012-05,2323
2,2012-07,2179
3,2012-04,2155
4,2012-08,2075
5,2012-06,1993
6,2012-10,1937
7,2016-08,1882
8,2016-05,1829
9,2016-06,1827



--- top 10 values: town ---


,town,count
0,JURONG WEST,7573
1,WOODLANDS,7399
2,TAMPINES,6728
3,BEDOK,6071
4,YISHUN,5937
5,SENGKANG,5896
6,HOUGANG,4735
7,ANG MO KIO,4558
8,CHOA CHU KANG,3928
9,BUKIT BATOK,3773


...

,street_name,count
0,YISHUN RING RD,1629
1,BEDOK RESERVOIR RD,1264
2,ANG MO KIO AVE 10,1188
3,ANG MO KIO AVE 3,1059
4,HOUGANG AVE 8,844
5,JURONG WEST ST 65,777
6,CHOA CHU KANG CRES,737
7,BEDOK NTH RD,731
8,BEDOK NTH ST 3,708
9,PUNGGOL CTRL,644



--- top 10 values: storey_range ---


,storey_range,count
0,04 TO 06,21214
1,07 TO 09,18765
2,01 TO 03,17466
3,10 TO 12,16028
4,13 TO 15,6704
5,16 TO 18,2706
6,01 TO 05,2700
7,06 TO 10,2474
8,11 TO 15,1259
9,19 TO 21,1156



--- top 10 values: flat_model ---


,flat_model,count
0,Model A,26447
1,Improved,24117
2,New Generation,16495
3,Premium Apartment,8314
4,Simplified,5152
5,Apartment,3701
6,Standard,3458
7,Maisonette,2574
8,Model A2,1415
9,DBSS,277


### 3.4 Flagged columns

Three flag types are emitted (config in `src/config.py`):

* **`high_null`** — null fraction > `PROFILE_HIGH_NULL_THRESHOLD` (50%).
* **`constant`** — only one distinct non-null value.
* **`whitespace`** — at least one string value differs from its `.str.strip()` form.

**Expected finding on this master:** only `remaining_lease` and `remaining_lease_years_original` should appear, both flagged `high_null` at ~60%. This is the vintage gap, not a defect — the pre-2015 source files don't carry the column at all (see the schema-by-vintage table in `README.md`). Stage 4 (validate) will treat this as known-absent rather than failing rows on it.

In [10]:
display(report.flags)

,column,flag,detail
0,remaining_lease,high_null,59.9% nulls
1,remaining_lease_years_original,high_null,59.9% nulls


## Next steps

The remaining stages of Part 1 (still pending):

* **§4 Validate.** Hand-rolled rule functions deriving allowed sets from the master's statistical properties (Date, Town, Flat Type, Flat Model, storey_range), plus numeric bounds and null-key checks. Failed rows go to `data/failed/validation_failures.csv`.
* **§5 Clean.** Dedupe on the composite key (highest `resale_price` wins), recompute `remaining_lease` as of today, flag anomalous prices via asymmetric IQR on price-per-sqm grouped by `(town, flat_type, year)`. Output: `data/cleaned/hdb_resale_cleaned.{csv,parquet}`.

**Out of scope (this pass):** Resale Identifier column, hashed identifier, `data/transformed/`, `data/hashed/`, Part 2 architecture diagrams.